In [1]:
# =============================================================================
# KAGGLE NOTEBOOK: REVIEWER METRICS EXTRACTION (PAPER 1 - SMCNN)
# =============================================================================
# Instructions:
# 1. Create a new Kaggle Notebook.
# 2. Add the datasets to your Kaggle environment:
#    - Indian Pines
#    - KSC (Kennedy Space Center)
# 3. Copy and paste this entire script into a cell.
# 4. Ensure GPU is turned on and run the cell.
# 5. It will print the exact F1 metrics, Confusion Matrix Deltas, and KSC metrics needed.
# =============================================================================

import os
import time
import numpy as np
import scipy.io as sio
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, f1_score

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import random

# Ensure exact reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# =============================================================================
# DATA LOADERS & PREPROCESSING
# =============================================================================

# UPDATE THESE PATHS TO YOUR EXACT KAGGLE DATASET PATHS
DATASET_INFO = {
    'IndianPines': {
        'data_url': 'http://www.ehu.eus/ccwintco/uploads/6/67/Indian_pines_corrected.mat',
        'gt_url':   'http://www.ehu.eus/ccwintco/uploads/c/c4/Indian_pines_gt.mat',
        'data_key': 'indian_pines_corrected',
        'gt_key':   'indian_pines_gt',
        'target_names': [
            'Alfalfa', 'Corn-notill', 'Corn-mintill', 'Corn',
            'Grass-pasture', 'Grass-trees', 'Grass-pasture-mowed',
            'Hay-windrowed', 'Oats', 'Soybean-notill', 'Soybean-mintill',
            'Soybean-clean', 'Wheat', 'Woods',
            'Bldg-Grass-Tree-Drives', 'Stone-Steel-Towers'
        ]
    },
    'KSC': {
        'data_url': '/kaggle/input/datasets/sreevallimanda/ksc-hyperspectral/KSC.mat',
        'gt_url':   '/kaggle/input/datasets/sreevallimanda/ksc-hyperspectral/KSC_gt.mat',
        'data_key': 'KSC',
        'gt_key':   'KSC_gt',
        'target_names': [
            'Scrub', 'Willow swamp', 'CP hammock', 'CP/Oak',
            'Slash pine', 'Oak/Broadleaf', 'Hardwood swamp',
            'Graminoid marsh', 'Spartina marsh', 'Cattail marsh',
            'Salt marsh', 'Mud flats', 'Water'
        ]
    }
}

# If paths are missing, we download them as a fallback (for local testing)
import urllib.request
import ssl

ssl._create_default_https_context = ssl._create_unverified_context

def load_dataset(dataset_name):
    info = DATASET_INFO[dataset_name]
    data_url = info['data_url']
    gt_url = info['gt_url']
    
    data_path = f"{dataset_name}.mat"
    gt_path = f"{dataset_name}_gt.mat"
    
    # Handle direct Kaggle inputs
    if data_url.startswith('/kaggle'):
        data_path = data_url
        gt_path = gt_url
    else:
        # Download if it's an HTTP link and not present locally
        if not os.path.exists(data_path):
            print(f"Downloading {dataset_name} data from {data_url}...")
            urllib.request.urlretrieve(data_url, data_path)
        if not os.path.exists(gt_path):
            print(f"Downloading {dataset_name} GT from {gt_url}...")
            urllib.request.urlretrieve(gt_url, gt_path)

    data = sio.loadmat(data_path)[info['data_key']]
    gt = sio.loadmat(gt_path)[info['gt_key']]
    return data, gt

def apply_pca(X, num_components=30):
    newX = np.reshape(X, (-1, X.shape[2]))
    pca = PCA(n_components=num_components, whiten=True)
    newX = pca.fit_transform(newX)
    return np.reshape(newX, (X.shape[0], X.shape[1], num_components))

def create_patches(X, y, window_size=11):
    margin = int((window_size - 1) / 2)
    zero_paddedX = np.pad(X, ((margin, margin), (margin, margin), (0, 0)), mode='constant')
    patchesData = []
    patchesLabels = []
    
    for r in range(margin, zero_paddedX.shape[0] - margin):
        for c in range(margin, zero_paddedX.shape[1] - margin):
            if y[r-margin, c-margin] > 0:
                patch = zero_paddedX[r - margin:r + margin + 1, c - margin:c + margin + 1]
                patchesData.append(patch)
                patchesLabels.append(y[r-margin, c-margin] - 1)
                
    return np.array(patchesData).transpose((0, 3, 1, 2)), np.array(patchesLabels)

def split_data(X, y, train_ratio=0.05, seed=42):
    np.random.seed(seed)
    train_idx, test_idx = [], []
    for c in np.unique(y):
        idx = np.where(y == c)[0]
        np.random.shuffle(idx)
        train_size = max(5, int(len(idx) * train_ratio))
        train_idx.extend(idx[:train_size])
        test_idx.extend(idx[train_size:])
    np.random.shuffle(train_idx)
    np.random.shuffle(test_idx)
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

# =============================================================================
# MODEL ARCHITECTURES
# =============================================================================

class PlainBlock(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(in_channels)
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out + res)

class SEBlock(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(in_channels)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        reduced_dim = max(4, in_channels // reduction)
        self.fc1 = nn.Linear(in_channels, reduced_dim)
        self.fc_scale = nn.Linear(reduced_dim, in_channels)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        b, c, _, _ = out.size()
        z = self.global_pool(out).view(b, c)
        z = self.relu(self.fc1(z))
        scale = self.sigmoid(self.fc_scale(z)).view(b, c, 1, 1)
        out = out * scale
        return self.relu(out + res)

class SSMRB(nn.Module):
    def __init__(self, in_channels, reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(in_channels, in_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(in_channels)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        reduced_dim = max(4, in_channels // reduction)
        self.fc1 = nn.Linear(in_channels, reduced_dim)
        self.fc_scale = nn.Linear(reduced_dim, in_channels)
        self.fc_shift = nn.Linear(reduced_dim, in_channels)
        self.sigmoid = nn.Sigmoid()
        
        # Hooks for analysis
        self.pre_mod_mean = None
        self.post_mod_mean = None

    def forward(self, x):
        res = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        self.pre_mod_mean = out.mean(dim=(0, 2, 3)).detach().cpu().numpy()
        b, c, _, _ = out.size()
        z = self.global_pool(out).view(b, c)
        z = self.relu(self.fc1(z))
        scale = self.sigmoid(self.fc_scale(z)).view(b, c, 1, 1)
        shift = self.fc_shift(z).view(b, c, 1, 1)
        out = out * scale + shift
        self.post_mod_mean = out.mean(dim=(0, 2, 3)).detach().cpu().numpy()
        return self.relu(out + res)

class SMCNN(nn.Module):
    def __init__(self, num_classes, num_bands=30, mod_type='plain'):
        super().__init__()
        self.conv1 = nn.Conv2d(num_bands, 64, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU()
        
        if mod_type == 'plain':
            self.block1 = PlainBlock(64)
            self.block2 = PlainBlock(64)
        elif mod_type == 'se':
            self.block1 = SEBlock(64)
            self.block2 = SEBlock(64)
        elif mod_type == 'ssmrb':
            self.block1 = SSMRB(64)
            self.block2 = SSMRB(64)
            
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(64, num_classes)
        
    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.block1(x)
        x = self.block2(x)
        x = self.global_pool(x).view(x.size(0), -1)
        return self.fc(x)

# =============================================================================
# TRAINING ENGINE
# =============================================================================

def train_and_evaluate(model, train_loader, test_loader, epochs=100, patience=20):
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5)
    criterion = nn.CrossEntropyLoss()
    
    best_oa = 0
    best_model_state = None
    epochs_no_improve = 0
    model.to(device)
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                outputs = model(X_batch)
                _, preds = torch.max(outputs, 1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())
                
        oa = accuracy_score(all_labels, all_preds)
        scheduler.step(oa)
        
        if oa > best_oa:
            best_oa = oa
            best_model_state = model.state_dict().copy()
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            
        if epochs_no_improve >= patience:
            break
            
    model.load_state_dict(best_model_state)
    
    # Final evaluation on best model
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
            
    return np.array(all_preds), np.array(all_labels)

# =============================================================================
# EXPERIMENT RUNNER
# =============================================================================

def run_experiments():
    seeds = [42, 100, 200, 300, 2024, 1, 7, 13, 77, 999]
    datasets = ['IndianPines', 'KSC']
    mod_types = ['plain', 'se', 'ssmrb']
    
    results = {}
    conf_matrices = {m: [] for m in mod_types}
    
    for ds in datasets:
        results[ds] = {m: {'oa': [], 'aa': [], 'macro_f1': [], 'min_f1': []} for m in mod_types}
        print(f"\n========================================")
        print(f"Loading and processing {ds}...")
        print(f"========================================")
        X, y = load_dataset(ds)
        X_pca = apply_pca(X, 30)
        X_pat, y_pat = create_patches(X_pca, y, window_size=11)
        num_classes = len(np.unique(y_pat))
        
        # Minority classes: define as having < 500 samples in the whole dataset
        class_counts = [np.sum(y_pat == i) for i in range(num_classes)]
        minority_idx = [i for i, c in enumerate(class_counts) if c < 500]
        
        for seed in seeds:
            print(f"\n--- Seed {seed} ---")
            X_tr, X_te, y_tr, y_te = split_data(X_pat, y_pat, seed=seed)
            
            train_ds = TensorDataset(torch.tensor(X_tr, dtype=torch.float32), torch.tensor(y_tr, dtype=torch.long))
            test_ds = TensorDataset(torch.tensor(X_te, dtype=torch.float32), torch.tensor(y_te, dtype=torch.long))
            train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
            test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)
            
            for mod in mod_types:
                set_seed(seed)
                model = SMCNN(num_classes, num_bands=30, mod_type=mod)
                preds, true = train_and_evaluate(model, train_loader, test_loader, epochs=100, patience=20)
                
                oa = accuracy_score(true, preds)
                report = classification_report(true, preds, output_dict=True, zero_division=0)
                aa = np.mean([report[str(i)]['recall'] for i in range(num_classes)])
                macro_f1 = f1_score(true, preds, average='macro')
                
                f1_scores = [report[str(i)]['f1-score'] for i in range(num_classes)]
                min_f1 = np.mean([f1_scores[i] for i in minority_idx])
                
                results[ds][mod]['oa'].append(oa * 100)
                results[ds][mod]['aa'].append(aa * 100)
                results[ds][mod]['macro_f1'].append(macro_f1 * 100)
                results[ds][mod]['min_f1'].append(min_f1 * 100)
                
                print(f"[{mod.upper()}] OA: {oa*100:.2f} | AA: {aa*100:.2f} | Macro F1: {macro_f1*100:.2f}")
                
                # Capture specific artifacts for Indian Pines, Seed 42
                if ds == 'IndianPines' and seed == 42:
                    conf_matrices[mod] = confusion_matrix(true, preds)
                    if mod == 'ssmrb':
                        pre_s = model.block1.pre_mod_mean
                        post_s = model.block1.post_mod_mean
                        plt.figure(figsize=(8,4))
                        plt.plot(pre_s, label='Before Modulation (SE+Shift)', alpha=0.7)
                        plt.plot(post_s, label='After SSMRB', alpha=0.7)
                        plt.legend()
                        plt.title('Activation Mean Shift via Beta Parameter')
                        plt.savefig('beta_verification_shift.png', dpi=300)
                        plt.close()
                        print("\n==> Saved beta_verification_shift.png (Verification plot)")
                        
    print("\n\n" + "="*50)
    print("FINAL 5-SEED RESULTS FOR REVIEWER TABLES")
    print("="*50)
    for ds in datasets:
        print(f"\n--- {ds} ---")
        for mod in mod_types:
            r = results[ds][mod]
            print(f"{mod.upper():>7}: OA={np.mean(r['oa']):.2f}±{np.std(r['oa']):.2f}  AA={np.mean(r['aa']):.2f}±{np.std(r['aa']):.2f}  MacroF1={np.mean(r['macro_f1']):.2f}±{np.std(r['macro_f1']):.2f}  MinF1={np.mean(r['min_f1']):.2f}±{np.std(r['min_f1']):.2f}")
                  
    print("\n\n" + "="*50)
    print("INDIAN PINES DELTA CONFUSION MATRIX (SE vs SSMRB, Seed 42)")
    print("="*50)
    se_cm = conf_matrices['se']
    ssmrb_cm = conf_matrices['ssmrb']
    
    se_acc = se_cm.diagonal() / se_cm.sum(axis=1) * 100
    ssmrb_acc = ssmrb_cm.diagonal() / ssmrb_cm.sum(axis=1) * 100
    delta = ssmrb_acc - se_acc
    
    target_names = DATASET_INFO['IndianPines']['target_names']
    print(f"{'Class Name':<25} | {'SE (%)':<8} | {'SSMRB (%)':<9} | {'Delta':<8}")
    print("-" * 57)
    for i in range(len(delta)):
        print(f"{target_names[i]:<25} | {se_acc[i]:>6.2f}   | {ssmrb_acc[i]:>7.2f}   | {delta[i]:>+6.2f}")

if __name__ == '__main__':
    run_experiments()


Using device: cuda

Loading and processing IndianPines...

--- Seed 42 ---
[PLAIN] OA: 95.86 | AA: 96.28 | Macro F1: 95.78
[SE] OA: 95.76 | AA: 93.19 | Macro F1: 94.67
[SSMRB] OA: 96.67 | AA: 97.17 | Macro F1: 96.14

==> Saved beta_verification_shift.png (Verification plot)

--- Seed 100 ---
[PLAIN] OA: 96.57 | AA: 95.94 | Macro F1: 93.24
[SE] OA: 96.45 | AA: 95.74 | Macro F1: 94.19
[SSMRB] OA: 95.36 | AA: 96.08 | Macro F1: 93.53

--- Seed 200 ---
[PLAIN] OA: 95.70 | AA: 96.45 | Macro F1: 92.44
[SE] OA: 96.28 | AA: 96.52 | Macro F1: 96.75
[SSMRB] OA: 95.96 | AA: 95.65 | Macro F1: 94.42

--- Seed 300 ---
[PLAIN] OA: 96.81 | AA: 96.92 | Macro F1: 97.24
[SE] OA: 96.97 | AA: 97.04 | Macro F1: 96.67
[SSMRB] OA: 95.90 | AA: 93.75 | Macro F1: 94.04

--- Seed 2024 ---
[PLAIN] OA: 96.71 | AA: 92.34 | Macro F1: 91.84
[SE] OA: 96.26 | AA: 95.57 | Macro F1: 94.76
[SSMRB] OA: 95.85 | AA: 93.33 | Macro F1: 92.35

--- Seed 1 ---
[PLAIN] OA: 96.97 | AA: 96.61 | Macro F1: 96.53
[SE] OA: 96.55 | AA: 97.